## Prerequisites

Before proceeding, the following components should already be available:

- Understanding of the Interest Rate Swap lifecycle
- Trade booking process
- InterestRateSwap class
- Trade validation framework
- Object-oriented representation of financial products

These components were developed in Notebook 1 and provide the foundation for schedule generation.

## Objective

The primary objective of this notebook is to construct the contractual payment schedule of a vanilla fixed-for-floating Interest Rate Swap.

Once a trade has been successfully captured, the trading system must determine the sequence of future contractual events associated with that transaction. These events define when interest accrues, when payments are exchanged and how the contract evolves throughout its lifetime.

Unlike pricing, schedule generation does not calculate the monetary value of future cashflows. Instead, it establishes the timeline upon which every downstream analytical process depends.

By the end of this notebook, the system will be capable of generating the complete schedule of coupon periods from the effective date to the maturity date using the contractual attributes captured during trade booking.

## Position within the Trade Lifecycle

Trade booking records the economic terms agreed between two counterparties.

However, recording a trade is only the beginning of its lifecycle.

Once the trade enters the bank's trading platform, multiple downstream systems immediately require information about future contractual events. Pricing engines need payment schedules to calculate present values. Treasury systems forecast future liquidity requirements. Settlement systems determine payment obligations. Market risk engines analyse future exposures, while accounting systems calculate accruals and financial reporting adjustments.

Consequently, schedule generation represents one of the earliest and most important stages of post-trade processing.

Rather than calculating prices, the objective is to answer a fundamental business question:

> **When do the contractual obligations of this trade occur?**

## Trade Lifecycle Workflow

The simplified workflow of an Interest Rate Swap within an enterprise trading platform is illustrated below.

```

```text
Trader Executes Trade
          │
          ▼
Trade Booking
          │
          ▼
Trade Validation
          │
          ▼
Trade Repository
          │
          ▼
Cashflow Schedule Generation
          │
 ┌────────┼─────────┬───────────┬────────────┐
 ▼        ▼         ▼           ▼            ▼
Pricing  Settlement Treasury  Accounting  Market Risk
```

```markdown

This notebook focuses exclusively on the **Cashflow Schedule Generation** stage.

The resulting schedule will become a shared input for every downstream analytical module developed throughout this project.

# Why Schedule Generation Comes Before Pricing

A common misconception is that pricing is the first analytical step after trade booking.

In reality, pricing cannot begin until the contractual timeline of the trade has been established.

For every coupon period, the pricing engine must know:

- when interest starts accruing,
- when interest stops accruing,
- when payment is exchanged,
- how long the accrual period lasts.

Without this information, neither coupon amounts nor discounting calculations can be performed.

For this reason, schedule generation forms the foundation upon which pricing, risk measurement and valuation adjustments are built.

# Understanding an Interest Rate Swap Timeline

Unlike a bond that may pay a fixed coupon according to a predefined issuance schedule, an Interest Rate Swap consists of two streams of contractual cashflows exchanged periodically between counterparties.

Each coupon period contains three important dates:

- Accrual Start Date
- Accrual End Date
- Payment Date

These dates determine the period over which interest accrues and the date on which the resulting payment obligation is settled.

The monetary amount associated with each payment will be calculated in later notebooks. The objective of this notebook is solely to construct these contractual periods.

# Example Timeline

Consider a five-year Interest Rate Swap with semi-annual payments.

```text
Effective Date
15-Jan-2025
      │
      ▼
15-Jul-2025
      │
      ▼
15-Jan-2026
      │
      ▼
15-Jul-2026
      │
      ▼
15-Jan-2027
      │
      ▼
...
      │
      ▼
15-Jan-2030
Maturity Date
```

Each interval represents one coupon period.

Every coupon period will eventually generate:

- a year fraction,
- a coupon payment,
- a discount factor,
- a present value,
- and a contribution to the overall value of the Interest Rate Swap.

# Software Design

Notebook 1 introduced the `InterestRateSwap` class, which models the contractual attributes agreed between counterparties.

The next step is to represent each coupon period as an independent business object.

Rather than storing payment dates as unrelated lists, each contractual period will be modelled as a separate `Cashflow` object.

This design improves readability, simplifies testing and reflects the architecture commonly adopted within enterprise pricing and risk libraries.

The relationship between these business objects can be represented as follows.

```text
InterestRateSwap
        │
        ├───────────────┐
        │               │
        ▼               ▼
 Cashflow 1        Cashflow 2
        │               │
        ▼               ▼
 Cashflow 3        Cashflow 4
        │
       ...
```

The Interest Rate Swap represents the entire financial contract, while each Cashflow object represents one contractual coupon period within that agreement.

# Modelling a Cashflow

The payment schedule generated for an Interest Rate Swap consists of multiple contractual periods.

Each period represents an independent business event with its own accrual interval, payment date and calculation parameters.

Rather than storing these values as unrelated lists or tables, each contractual period will be represented as an individual software object.

This object-oriented design provides several advantages.

- Each cashflow can be validated independently.
- Additional financial attributes can be added without modifying the entire schedule.
- Pricing and risk calculations can operate directly on individual cashflows.
- The implementation closely reflects the architecture adopted by enterprise trading and risk platforms.

Initially, the Cashflow object will contain scheduling information only.

Financial quantities such as coupon amounts, discount factors, present values and sensitivities will be introduced in subsequent notebooks.

# Information Stored within a Cashflow

Every coupon period contains the information required to describe one contractual payment interval.

| Attribute | Description |
|-----------|-------------|
| Period Number | Sequential coupon number |
| Accrual Start Date | Beginning of the interest accrual period |
| Accrual End Date | End of the interest accrual period |
| Payment Date | Date on which payment is exchanged |
| Year Fraction | Fraction of the year represented by the accrual period |
| Leg Type | Fixed Leg or Floating Leg |

At this stage, the cashflow object describes **when** a contractual event occurs.

It does not yet determine **how much** will be paid.

The monetary amount depends upon interest rates, day count conventions and discounting methodologies that will be developed later in the project.

# Software Design

The relationship between the business objects is now expanded.

```text
InterestRateSwap
        │
        │ contains
        ▼
+----------------------+
|      Cashflow        |
+----------------------+
| Period Number        |
| Accrual Start Date   |
| Accrual End Date     |
| Payment Date         |
| Year Fraction        |
| Leg Type             |
+----------------------+
```

One Interest Rate Swap consists of multiple Cashflow objects.

As the project evolves, every pricing and risk calculation will operate on these individual cashflows before being aggregated back to the swap level.

In [36]:
import os
import sys

project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [37]:
from datetime import date

from src.products.interest_rate_swap import InterestRateSwap
from src.products.cashflow import Cashflow
from src.engines.schedule_generator import ScheduleGenerator

# Why Another Class?

An Interest Rate Swap represents the complete contractual agreement between two counterparties.

A Cashflow represents only one contractual period within that agreement.

Separating these responsibilities improves modularity and allows each component to evolve independently.

For example, future notebooks will extend the Cashflow object by adding:

- coupon rate,
- notional amount,
- discount factor,
- present value,
- DV01 contribution,
- expected exposure,
- valuation adjustments.

The InterestRateSwap object therefore acts as a container that owns multiple Cashflow objects rather than attempting to perform every calculation itself.

# Date Arithmetic in Financial Systems

Dates play a central role in every financial contract.

For an Interest Rate Swap, every future coupon payment depends upon accurately determining the boundaries of each accrual period. Although this appears straightforward, financial date calculations are considerably more complex than ordinary calendar arithmetic.

Unlike days, months do not contain a fixed number of calendar days.

For example,

- January contains 31 days,
- February contains 28 or 29 days,
- April contains 30 days.

Consequently, adding a fixed number of days does not necessarily produce the next contractual payment date.

Financial contracts therefore define payment schedules in terms of **tenors** such as:

- 1 Month
- 3 Months
- 6 Months
- 1 Year

rather than fixed numbers of days.

Enterprise trading systems must therefore perform calendar-aware date calculations that correctly interpret contractual frequencies while preserving the intended payment schedule.

# Why Calendar Arithmetic Matters

Consider a semi-annual Interest Rate Swap with an effective date of **15 January 2025**.

A human immediately recognises that the next coupon date should be:

```

```text
15-Jan-2025

↓

15-Jul-2025
```

```markdown

However, computers do not naturally understand the concept of "six months."

They understand numbers.

A naive implementation might attempt to add 180 days.

Although this appears reasonable, six calendar months are not always equal to 180 days.

As a result, repeatedly adding a fixed number of days gradually shifts contractual payment dates away from the legally agreed schedule.

Enterprise trading systems therefore generate schedules by adding calendar months rather than calendar days.

# Financial Calendar Conventions

Professional pricing libraries must also account for market conventions that influence payment schedules.

These include:

- Business day conventions
- Holiday calendars
- End-of-month adjustments
- Stub periods
- Leap years
- Currency-specific settlement calendars

Implementing every market convention is beyond the scope of this notebook.

Instead, the objective is to build a simplified scheduling engine capable of generating regular payment periods for vanilla Interest Rate Swaps.

Additional market conventions will be incorporated incrementally in later notebooks as the platform evolves. 

# Choosing an Appropriate Date Library

Python's standard `datetime` module provides excellent support for working with dates.

However, it does not directly support operations such as adding calendar months.

For example, Python does not provide a native operation equivalent to:

```python
next_date = current_date + 6 months
```

To perform calendar-aware month arithmetic, this project uses the `dateutil` package and its `relativedelta` class.

Unlike adding a fixed number of days, `relativedelta` correctly interprets contractual tenors expressed in months and years, making it well suited for financial scheduling applications.

# Schedule Generation Engine

The InterestRateSwap object stores the contractual terms of the transaction.

However, determining future payment periods is a separate responsibility.

Rather than embedding scheduling logic directly within the trade object, this project introduces a dedicated ScheduleGenerator component.

The scheduling engine receives an Interest Rate Swap as input and produces a sequence of Cashflow objects representing the contractual payment schedule.

This separation follows the principle of single responsibility.

- The trade represents the contract.
- The scheduling engine constructs the contractual timeline.
- Pricing engines will value the generated cashflows.
- Risk engines will analyse the resulting exposures.

This modular architecture closely resembles enterprise trading and risk platforms, where different analytical components operate on the same underlying trade representation.

# System Architecture

The objective of this project is not simply to price an Interest Rate Swap.

Instead, the objective is to build a modular enterprise risk analytics platform that reflects the architecture adopted within modern investment banks.

Rather than concentrating all business logic within a single software class, individual components are designed to perform specialised responsibilities.

This separation improves maintainability, encourages code reuse and allows analytical modules to evolve independently as the platform grows.

# High-Level Architecture

The platform consists of four major layers.

```text
                Enterprise Risk Analytics Platform
┌──────────────────────────────────────────────────────────────┐
│                      Business Objects                        │
│                                                              │
│   Portfolio                                                  │
│        │                                                     │
│        ▼                                                     │
│   InterestRateSwap                                           │
│        │                                                     │
│        ▼                                                     │
│     Cashflow                                                 │
└──────────────────────────────────────────────────────────────┘

                    │
                    ▼

┌──────────────────────────────────────────────────────────────┐
│                     Market Data Layer                        │
│                                                              │
│   Yield Curves                                               │
│   FX Rates                                                   │
│   Holiday Calendars                                          │
│   Market Fixings                                             │
└──────────────────────────────────────────────────────────────┘

                    │
                    ▼

┌──────────────────────────────────────────────────────────────┐
│                    Analytical Engines                        │
│                                                              │
│   Schedule Generator                                         │
│   Pricing Engine                                             │
│   Sensitivity Engine                                         │
│   VaR Engine                                                 │
│   FRTB Engine                                                │
│   XVA Engine                                                 │
└──────────────────────────────────────────────────────────────┘

                    │
                    ▼

┌──────────────────────────────────────────────────────────────┐
│                         Outputs                              │
│                                                              │
│   Present Value                                              │
│   Risk Measures                                              │
│   Regulatory Capital                                         │
│   AI Commentary                                               │
└──────────────────────────────────────────────────────────────┘
```

# Business Objects

Business objects represent contractual information agreed between counterparties.

They do not perform analytical calculations.

Instead, they provide the structured data required by downstream analytical engines.

Throughout this project, three primary business objects will be developed.

| Business Object | Responsibility |
|-----------------|----------------|
| Portfolio | Collection of financial trades |
| InterestRateSwap | Representation of a single contractual agreement |
| Cashflow | Representation of one contractual payment period |

These objects evolve throughout the project as additional financial attributes become available.

# Analytical Engines

Analytical engines consume business objects and produce financial calculations.

Each engine has a clearly defined responsibility.

| Engine | Responsibility |
|----------|----------------|
| Schedule Generator | Construct contractual payment periods |
| Pricing Engine | Calculate present value |
| Sensitivity Engine | Calculate DV01, PV01 and other Greeks |
| Market Risk Engine | Historical VaR and stress testing |
| FRTB Engine | Regulatory capital calculations |
| XVA Engine | Counterparty valuation adjustments |

This modular architecture allows each analytical component to be developed, tested and maintained independently.

# Project Evolution

Each notebook expands the capabilities of the platform by introducing one additional component.

| Notebook | Component Introduced |
|----------|----------------------|
| 1 | InterestRateSwap |
| 2 | Cashflow & Schedule Generator |
| 3 | Yield Curve |
| 4 | Pricing Engine |
| 5 | Sensitivity Engine |
| 6 | Portfolio |
| 7 | Market Risk Engine |
| 8 | FRTB GIRR Engine |
| 9 | FRTB FX Engine |
| 10 | Counterparty Exposure |
| 11 | Hull-White Interest Rate Model |
| 12 | XVA Engine |
| 13 | AI Risk Commentary |

By the conclusion of the project, the platform will be capable of progressing from trade capture to pricing, market risk, regulatory capital and valuation adjustments using a consistent software architecture.

# Implementing the Schedule Generator

The ScheduleGenerator is the first analytical engine developed within the platform.

Its responsibility is to transform the contractual information contained within an InterestRateSwap into a sequence of Cashflow objects.

The engine does not perform any pricing calculations.

Instead, it constructs the contractual payment schedule that serves as the foundation for all subsequent valuation and risk analytics.

The generated cashflows will later be consumed by pricing, sensitivity and regulatory capital engines without requiring any modification to the original trade representation.

# Designing the Schedule Generator

The ScheduleGenerator is responsible for constructing the contractual payment schedule of an Interest Rate Swap.

Rather than embedding scheduling logic within the InterestRateSwap class, this functionality is implemented as a dedicated analytical engine.

The engine receives a trade as input, generates the sequence of contractual payment periods and returns a collection of Cashflow objects.

Separating schedule generation from the trade object improves modularity and allows future scheduling rules to evolve independently from the trade representation.

# Converting Payment Frequencies into Calendar Months

Financial contracts describe payment frequencies using business terminology such as Quarterly, Semi-Annual and Annual.

However, calendar arithmetic requires numerical month intervals.

Before constructing the payment schedule, the scheduling engine must translate contractual frequencies into the corresponding number of calendar months.

Encapsulating this translation within a dedicated helper method keeps the scheduling algorithm simple and improves maintainability.

# Designing the Scheduling Algorithm

Generating a payment schedule is an iterative process.

The algorithm begins at the effective date of the Interest Rate Swap and repeatedly advances by the contractual payment frequency until the maturity date is reached.

For each coupon period, the scheduling engine records:

- Accrual Start Date
- Accrual End Date
- Payment Date
- Year Fraction
- Leg Type

Each coupon period is represented as a Cashflow object.

The resulting collection of Cashflow objects forms the contractual payment schedule of the trade.

# Scheduling Algorithm

```text
Effective Date
      │
      ▼
Current Date

      │
      ▼
Add Payment Frequency

      │
      ▼
Create Cashflow Object

      │
      ▼
Store Cashflow

      │
      ▼
Current Date = Next Date

      │
      ▼
Reached Maturity?

No ─────────────► Continue

Yes ────────────► Finish
```

In [38]:
swap = InterestRateSwap(
    trade_id="IRS001",
    counterparty="ABC Bank",
    currency="USD",
    notional=10_000_000,
    fixed_rate=0.045,
    floating_index="SOFR",
    pay_fixed=True,
    effective_date=date(2025, 1, 15),
    maturity_date=date(2030, 1, 15),
    payment_frequency="Semi-Annual",
    day_count_convention="ACT/360",
    status="Active",
)

In [39]:
generator = ScheduleGenerator()

cashflows = generator.generate(swap)

In [40]:
for cashflow in cashflows:
    cashflow.summary()

Trade ID      : IRS001
Coupon Period : 1
Accrual Start : 2025-01-15
Accrual End   : 2025-07-15
Payment Date  : 2025-07-15
Year Fraction : 0.500000
Leg Type      : Fixed
Trade ID      : IRS001
Coupon Period : 2
Accrual Start : 2025-07-15
Accrual End   : 2026-01-15
Payment Date  : 2026-01-15
Year Fraction : 0.500000
Leg Type      : Fixed
Trade ID      : IRS001
Coupon Period : 3
Accrual Start : 2026-01-15
Accrual End   : 2026-07-15
Payment Date  : 2026-07-15
Year Fraction : 0.500000
Leg Type      : Fixed
Trade ID      : IRS001
Coupon Period : 4
Accrual Start : 2026-07-15
Accrual End   : 2027-01-15
Payment Date  : 2027-01-15
Year Fraction : 0.500000
Leg Type      : Fixed
Trade ID      : IRS001
Coupon Period : 5
Accrual Start : 2027-01-15
Accrual End   : 2027-07-15
Payment Date  : 2027-07-15
Year Fraction : 0.500000
Leg Type      : Fixed
Trade ID      : IRS001
Coupon Period : 6
Accrual Start : 2027-07-15
Accrual End   : 2028-01-15
Payment Date  : 2028-01-15
Year Fraction : 0.500000
Leg Type

# Summary

In this notebook, the first analytical engine of the platform was developed.

The ScheduleGenerator transforms the contractual terms of an InterestRateSwap into a sequence of Cashflow objects representing the contractual payment schedule.

Several important software engineering principles were introduced:

- Separation of business objects from analytical engines
- Composition through collections of Cashflow objects
- Calendar-aware date arithmetic using relativedelta
- Incremental construction of reusable analytical components

The generated payment schedule forms the foundation for all subsequent valuation and risk calculations.

The next notebook introduces yield curve construction, enabling the generated cashflows to be discounted and valued.